In [49]:
from dotenv import load_dotenv
import psycopg2
from psycopg2.extras import execute_values
import pandas as pd 
import numpy as np
import os
import re

In [50]:
# Improvements to the script 
# - Need a way to investigate top rated golfers 
# want to see a print out of top golfers and their assigned tier 
# might need to put a min threshold on the amount of events, current analysis bias towards less events (done must at least 5)
# want to have a print out of past standings for an event 
# want to see the x last finishes for a golfer by search 
# eventually need to track results - success of picks 

In [51]:
# Load environment variables from .env file
load_dotenv()

# Database configuration from environment variables
db_config = {
    'host': os.getenv('DB_HOST'),
    'database': os.getenv('DB_NAME'),
    'user': os.getenv('DB_USER'),
    'password': os.getenv('DB_PASSWORD'),
    'port': os.getenv('DB_PORT', '5432')
}
schema = os.getenv('DB_SCHEMA')

# Test the connection
try:
    conn = psycopg2.connect(**db_config)
    conn.close()
    print("✅ Database connection successful!")
    print(f"   Connected to: {db_config['database']} on {db_config['host']}")
except Exception as e:
    print(f"❌ Connection failed: {e}")

✅ Database connection successful!
   Connected to: brdb on localhost


In [52]:
# Create a cursor
conn = psycopg2.connect(**db_config)
cur = conn.cursor()

# Example query
query = ''' select a.player , a.pos , a.to_par 
, a.year , a.week_of_season , a.tournament , a.course
, b.avg , case when b.avg is null then -2 else b.avg end Updated_Avg
, b.total_sg_t , b.total_sg_t2g , b.total_sg_p , measured_rounds 
from pga_stats.Leaderboard_data a 
left join pga_stats.sg_data b
on a.player = b.player 
and a.year = b.year 
and a.week_of_season = b.week_of_season
'''

# Execute the query
cur.execute(query)

# Fetch results
rows = cur.fetchall()

colnames = [desc[0] for desc in cur.description]

# Load into DataFrame
df = pd.DataFrame(rows, columns=colnames)

# Close connections
#cur.close()
#conn.close()

In [53]:
# Sort data
df = df.sort_values(by=['player', 'year', 'week_of_season'])

# Group by golfer and compute rolling averages
df['SG_last_1'] = df.groupby('player')['updated_avg'].transform(lambda x: x.rolling(1, min_periods=1).mean())
df['SG_last_2'] = df.groupby('player')['updated_avg'].transform(lambda x: x.rolling(2, min_periods=1).mean())
df['SG_last_3'] = df.groupby('player')['updated_avg'].transform(lambda x: x.rolling(3, min_periods=1).mean())
df['SG_last_5'] = df.groupby('player')['updated_avg'].transform(lambda x: x.rolling(5, min_periods=1).mean())
df['SG_last_8'] = df.groupby('player')['updated_avg'].transform(lambda x: x.rolling(8, min_periods=1).mean())
df['SG_last_13'] = df.groupby('player')['updated_avg'].transform(lambda x: x.rolling(13, min_periods=1).mean())
df['SG_last_21'] = df.groupby('player')['updated_avg'].transform(lambda x: x.rolling(21, min_periods=1).mean())
df['SG_last_34'] = df.groupby('player')['updated_avg'].transform(lambda x: x.rolling(34, min_periods=1).mean())
df['SG_last_55'] = df.groupby('player')['updated_avg'].transform(lambda x: x.rolling(55, min_periods=1).mean())
df['SG_last_89'] = df.groupby('player')['updated_avg'].transform(lambda x: x.rolling(89, min_periods=1).mean())


In [54]:
df = df.round(3)
display(df)

,player,pos,to_par,year,week_of_season,tournament,course,avg,updated_avg,total_sg_t,...,SG_last_1,SG_last_2,SG_last_3,SG_last_5,SG_last_8,SG_last_13,SG_last_21,SG_last_34,SG_last_55,SG_last_89
0,A.J. Ewart,CUT,0,2025,23.0,RBC Canadian Open,TPC Toronto at Osprey Valley,None,-2,None,...,-2.000,-2.000,-2.000,-2.000,-2.000,-2.000,-2.000,-2.000,-2.000,-2.000
1,A.J. Ewart,CUT,3,2026,2.0,Sony Open in Hawaii,Waialae Country Club,None,-2,None,...,-2.000,-2.000,-2.000,-2.000,-2.000,-2.000,-2.000,-2.000,-2.000,-2.000
2,A.J. Ewart,T44,-15,2026,3.0,The American Express,PGA West,-0.469,-0.469,-0.938,...,-0.469,-1.234,-1.490,-1.490,-1.490,-1.490,-1.490,-1.490,-1.490,-1.490
3,A.J. Ewart,T49,-5,2026,4.0,Farmers Insurance Open,Torrey Pines Golf Course,0.277,0.277,0.832,...,0.277,-0.096,-0.731,-1.048,-1.048,-1.048,-1.048,-1.048,-1.048,-1.048
4,A.J. Ewart,T28,-7,2026,6.0,WM Phoenix Open,TPC Scottsdale,0.924,0.924,3.694,...,0.924,0.600,0.244,-0.654,-0.654,-0.654,-0.654,-0.654,-0.654,-0.654
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19821,Zecheng Dou,T13,-11,2026,6.0,WM Phoenix Open,TPC Scottsdale,1.924,1.924,7.694,...,1.924,1.586,0.568,-0.066,-0.791,-0.934,-0.676,-1.009,-1.009,-1.009
1361,Ángel Cabrera,CUT,11,2025,15.0,Masters Tournament,Augusta National Golf Club,None,-2,None,...,-2.000,-2.000,-2.000,-2.000,-2.000,-2.000,-2.000,-2.000,-2.000,-2.000
6379,Étienne Papineau,CUT,4,2022,6.0,WM Phoenix Open,TPC Scottsdale,None,-2,None,...,-2.000,-2.000,-2.000,-2.000,-2.000,-2.000,-2.000,-2.000,-2.000,-2.000
6380,Étienne Papineau,CUT,1,2023,23.0,RBC Canadian Open,TPC Toronto at Osprey Valley,None,-2,None,...,-2.000,-2.000,-2.000,-2.000,-2.000,-2.000,-2.000,-2.000,-2.000,-2.000


In [55]:
player_view = df[df['player'].str.lower() == 'ben griffin']
display(player_view)

,player,pos,to_par,year,week_of_season,tournament,course,avg,updated_avg,total_sg_t,...,SG_last_1,SG_last_2,SG_last_3,SG_last_5,SG_last_8,SG_last_13,SG_last_21,SG_last_34,SG_last_55,SG_last_89
1786,Ben Griffin,CUT,-1,2022,8.0,Mexico Open at VidantaWorld,Vidanta Vallarta,None,-2,None,...,-2.000,-2.000,-2.000,-2.000,-2.000,-2.000,-2.000,-2.000,-2.000,-2.000
1787,Ben Griffin,4,-14,2022,31.0,Wyndham Championship,Sedgefield Country Club,2.598,2.598,10.391,...,2.598,0.299,0.299,0.299,0.299,0.299,0.299,0.299,0.299,0.299
1788,Ben Griffin,CUT,2,2022,35.0,Procore Championship,Silverado Resort,None,-2,None,...,-2.000,0.299,-0.467,-0.467,-0.467,-0.467,-0.467,-0.467,-0.467,-0.467
1789,Ben Griffin,T24,-9,2022,36.0,Sanderson Farms Championship,The Country Club of Jackson,None,-2,None,...,-2.000,-2.000,-0.467,-0.850,-0.850,-0.850,-0.850,-0.850,-0.850,-0.850
1790,Ben Griffin,T59,-6,2022,39.0,World Wide Technology Championship,El Cardonal at Diamante,None,-2,None,...,-2.000,-2.000,-2.000,-1.080,-1.080,-1.080,-1.080,-1.080,-1.080,-1.080
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1875,Ben Griffin,1,-29,2025,39.0,World Wide Technology Championship,El Cardonal at Diamante,None,-2,None,...,-2.000,0.898,0.871,1.033,0.432,1.185,0.582,0.401,0.242,0.041
1876,Ben Griffin,T19,-8,2026,2.0,Sony Open in Hawaii,Waialae Country Club,1.218,1.218,4.871,...,1.218,-0.391,1.005,1.002,0.835,1.023,0.679,0.411,0.300,0.025
1877,Ben Griffin,T24,-18,2026,3.0,The American Express,PGA West,0.791,0.791,1.582,...,0.791,1.005,0.003,0.924,1.184,0.863,0.812,0.382,0.300,0.057
1878,Ben Griffin,T37,-11,2026,5.0,ATT Pebble Beach Pro-Am,Pebble Beach Golf Links,0.125,0.125,0.500,...,0.125,0.458,0.711,0.786,0.912,0.674,0.913,0.444,0.310,0.080


In [56]:
latest = df.groupby('player').tail(1).reset_index(drop=True)

In [57]:
# Query to call updated datagolf ranks 

'''
conn = psycopg2.connect(
    dbname='brdb',
    user='brandon',
    password='',
    host='localhost',
    port='5432'
)
'''

cur = conn.cursor()

# Write your SQL query
query = "SELECT * FROM pga_stats.datagolf_ranks;"

# Load data into a DataFrame
df = pd.read_sql_query(query, conn)

# Close the connection
# conn.close()

# Now df contains your table data
print(df.head())

     id        player_name   primary_tour  dg_rank  owgr_rank  dg_index  \
0  1001  Scottie Scheffler       PGA Tour        1          1     3.155   
1  1002           Jon Rahm       LIV Golf        2         50     1.961   
2  1003    Tommy Fleetwood  DP World Tour        3          3     1.865   
3  1004       Rory McIlroy  DP World Tour        4          2     1.776   
4  1005     Russell Henley       PGA Tour        5          6     1.706   

                refresh_date  
0 2026-02-16 23:12:24.273811  
1 2026-02-16 23:12:24.273811  
2 2026-02-16 23:12:24.273811  
3 2026-02-16 23:12:24.273811  
4 2026-02-16 23:12:24.273811  


In [58]:
joined_df = pd.merge(latest, df, how='left', left_on='player', right_on='player_name')

In [59]:
joined_df

,player,pos,to_par,year,week_of_season,tournament,course,avg,updated_avg,total_sg_t,...,SG_last_34,SG_last_55,SG_last_89,id,player_name,primary_tour,dg_rank,owgr_rank,dg_index,refresh_date
0,A.J. Ewart,T28,-7,2026,6.0,WM Phoenix Open,TPC Scottsdale,0.924,0.924,3.694,...,-0.654,-0.654,-0.654,1299.0,A.J. Ewart,Other Tours,299.0,363.0,-0.637,2026-02-16 23:12:24.273811
1,A.J. Ewart (a),CUT,5,2022,23.0,RBC Canadian Open,TPC Toronto at Osprey Valley,None,-2,None,...,-2.000,-2.000,-2.000,NaN,NaN,NaN,NaN,NaN,NaN,NaT
2,Aaron Baddeley,T72,2,2025,31.0,Wyndham Championship,Sedgefield Country Club,-1.204,-1.204,-4.816,...,-1.136,-1.030,-1.087,NaN,NaN,NaN,NaN,NaN,NaN,NaT
3,Aaron Beverly,CUT,14,2022,7.0,The Genesis Invitational,The Riviera Country Club,None,-2,None,...,-2.000,-2.000,-2.000,NaN,NaN,NaN,NaN,NaN,NaN,NaT
4,Aaron Cockerill,CUT,4,2025,28.0,Genesis Scottish Open,The Renaissance Club,None,-2,None,...,-1.543,-1.543,-1.543,1404.0,Aaron Cockerill,DP World Tour,404.0,462.0,-0.947,2026-02-16 23:12:24.273811
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1183,Zack Fischer,CUT,-1,2025,27.0,John Deere Classic,TPC Deere Run,None,-2,None,...,-1.282,-1.282,-1.282,NaN,NaN,NaN,NaN,NaN,NaN,NaT
1184,Zander Lombard,CUT,2,2023,28.0,Genesis Scottish Open,The Renaissance Club,None,-2,None,...,-2.000,-2.000,-2.000,1371.0,Zander Lombard,DP World Tour,371.0,472.0,-0.869,2026-02-16 23:12:24.273811
1185,Zecheng Dou,T13,-11,2026,6.0,WM Phoenix Open,TPC Scottsdale,1.924,1.924,7.694,...,-1.009,-1.009,-1.009,1122.0,Zecheng Dou,Korn Ferry Tour,122.0,185.0,0.171,2026-02-16 23:12:24.273811
1186,Ángel Cabrera,CUT,11,2025,15.0,Masters Tournament,Augusta National Golf Club,None,-2,None,...,-2.000,-2.000,-2.000,NaN,NaN,NaN,NaN,NaN,NaN,NaT


In [60]:
# Percentile Ranks
sg_cols = [
    'SG_last_1', 'SG_last_2', 'SG_last_3', 'SG_last_5', 'SG_last_8',
    'SG_last_13', 'SG_last_21', 'SG_last_34', 'SG_last_55', 'SG_last_89', 'dg_index'
]

for col in sg_cols:
    joined_df[f'{col}_percentile'] = joined_df[col].rank(pct=True, ascending=True)

In [61]:
# If you want descending order (best performer = 100th percentile)
joined_df['owgr_rank_percentile'] = joined_df['owgr_rank'].rank(pct=True,ascending=False)
joined_df = joined_df.round(4)

In [62]:
joined_df

,player,pos,to_par,year,week_of_season,tournament,course,avg,updated_avg,total_sg_t,...,SG_last_3_percentile,SG_last_5_percentile,SG_last_8_percentile,SG_last_13_percentile,SG_last_21_percentile,SG_last_34_percentile,SG_last_55_percentile,SG_last_89_percentile,dg_index_percentile,owgr_rank_percentile
0,A.J. Ewart,T28,-7,2026,6.0,WM Phoenix Open,TPC Scottsdale,0.924,0.924,3.694,...,0.9386,0.8729,0.8687,0.8683,0.8636,0.8611,0.8577,0.8569,0.3224,0.2494
1,A.J. Ewart (a),CUT,5,2022,23.0,RBC Canadian Open,TPC Toronto at Osprey Valley,None,-2,None,...,0.3590,0.3295,0.3018,0.2883,0.2862,0.2862,0.2858,0.2858,NaN,NaN
2,Aaron Baddeley,T72,2,2025,31.0,Wyndham Championship,Sedgefield Country Club,-1.204,-1.204,-4.816,...,0.7168,0.6524,0.5976,0.6549,0.6515,0.7391,0.7626,0.7433,NaN,NaN
3,Aaron Beverly,CUT,14,2022,7.0,The Genesis Invitational,The Riviera Country Club,None,-2,None,...,0.3590,0.3295,0.3018,0.2883,0.2862,0.2862,0.2858,0.2858,NaN,NaN
4,Aaron Cockerill,CUT,4,2025,28.0,Genesis Scottish Open,The Renaissance Club,None,-2,None,...,0.3590,0.7151,0.6978,0.6625,0.6397,0.6364,0.6309,0.6279,0.1385,0.1688
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1183,Zack Fischer,CUT,-1,2025,27.0,John Deere Classic,TPC Deere Run,None,-2,None,...,0.7744,0.7753,0.7559,0.7306,0.7117,0.7045,0.6978,0.6944,NaN,NaN
1184,Zander Lombard,CUT,2,2023,28.0,Genesis Scottish Open,The Renaissance Club,None,-2,None,...,0.3590,0.3295,0.3018,0.2883,0.2862,0.2862,0.2858,0.2858,0.1990,0.1511
1185,Zecheng Dou,T13,-11,2026,6.0,WM Phoenix Open,TPC Scottsdale,1.924,1.924,7.694,...,0.9604,0.9394,0.8443,0.8123,0.8590,0.7782,0.7715,0.7656,0.7103,0.5668
1186,Ángel Cabrera,CUT,11,2025,15.0,Masters Tournament,Augusta National Golf Club,None,-2,None,...,0.3590,0.3295,0.3018,0.2883,0.2862,0.2862,0.2858,0.2858,NaN,NaN


In [63]:
# Pull leaderboard data
'''
conn = psycopg2.connect(
    dbname='brdb',
    user='brandon',
    password='',
    host='localhost',
    port='5432'
)
'''
cur = conn.cursor()

# Write your SQL query
query = '''select player , pos , to_par , official_money 
, year , tournament , week_of_season , course 
from pga_stats.leaderboard_data ;'''

# Load data into a DataFrame
df = pd.read_sql_query(query, conn)

# Close the connection
# conn.close()

# Sort data
df = df.sort_values(by=['player', 'year', 'week_of_season'])

In [64]:
# CONFIGURATION: Choose how to handle cuts
CUT_HANDLING = "penalty"  # Options: "penalty", "skip", "field_size"
CUT_PENALTY_POSITION = 80  # Position assigned to cuts when using "penalty" method

# Convert position to numeric, handling string positions like "T5", "T10", etc.
def clean_position(pos):
    if pd.isna(pos):
        return np.nan
    
    # Convert to string and handle tied positions
    pos_str = str(pos).upper().strip()
    
    # Handle cuts based on chosen method
    if pos_str == 'CUT' or pos_str == 'MC':  # MC = Missed Cut
        if CUT_HANDLING == "penalty":
            return CUT_PENALTY_POSITION  # Fixed penalty position
        elif CUT_HANDLING == "skip":
            return np.nan  # Skip cuts entirely - don't count in averages

    # Remove 'T' for tied positions and convert to integer
    if pos_str.startswith('T'):
        return int(pos_str[1:])
    else:
        try:
            return int(pos_str)
        except ValueError:
            return np.nan
        
# Apply position cleaning
df['position_numeric'] = df['pos'].apply(clean_position)

# Sort by player and then by week/year to get chronological order
df_sorted = df.sort_values(['player', 'year', 'week_of_season'], ascending=[True, True, True])

In [65]:
# Group by player and calculate rolling averages
results = []

# Get current date info for filtering last year's data
current_year = df['year'].max()  # Use the most recent year in the data
last_year_cutoff = current_year - 1  # Look at roughly last year of data


for player, group in df_sorted.groupby('player'):
    # Get all tournament entries (including cuts) in chronological order
    all_entries = group.dropna(subset=['pos']).copy()
    
    # Filter for last year's data for cut percentage calculation
    # recent_entries = all_entries[all_entries['year'] >= current_year] # this is the old version that was just pulling current year values but was supposed to be prior year
    recent_entries = all_entries[all_entries['year'] == last_year_cutoff]
    # Calculate cut percentage over the last year
    if len(recent_entries) > 0:
        cuts_made = len(recent_entries[~recent_entries['pos'].str.upper().isin(['CUT', 'MC'])])
        total_tournaments = len(recent_entries)
        cut_percentage = (cuts_made / total_tournaments) * 100
    else:
        cut_percentage = np.nan
        total_tournaments = 0
    
    # Get numeric positions for averaging (this excludes NaN values from cuts if using "skip" method)
    positions = group['position_numeric'].dropna().tolist()
    
    if len(positions) == 0:
        continue

    # 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144
    
    recent_positions = recent_entries[recent_entries['position_numeric'].notna()]['position_numeric'].tolist()


    top_5_count = len([pos for pos in recent_positions if pos <= 5])
    top_10_count = len([pos for pos in recent_positions if pos <= 10]) 
    top_20_count = len([pos for pos in recent_positions if pos <= 20])
    # Calculate average for last 3 games


    # Calculate percentages based on tournaments finished (not including cuts)
    num_recent_positions = len(recent_positions)
    if num_recent_positions > 0:
        top_5_percentage = (top_5_count / num_recent_positions) * 100
        top_10_percentage = (top_10_count / num_recent_positions) * 100
        top_20_percentage = (top_20_count / num_recent_positions) * 100
    else:
        top_5_percentage = 0.0
        top_10_percentage = 0.0
        top_20_percentage = 0.0

    last_1_avg = np.mean(positions[-1:]) if len(positions) >= 1 else np.nan
    last_3_avg = np.mean(positions[-3:]) if len(positions) >= 1 else np.nan
    
    # Calculate average for last 5 games
    last_5_avg = np.mean(positions[-5:]) if len(positions) >= 1 else np.nan
    
    # Calculate average for last 10 games
    last_10_avg = np.mean(positions[-10:]) if len(positions) >= 1 else np.nan

    # Count of games with valid positions (excludes cuts if using "skip" method)
    games_with_positions = len(positions)
    
    # Count of all tournament entries
    total_entries = len(all_entries)
    
    results.append({
        'player': player,
        'total_tournaments': total_entries,
        'games_with_positions': games_with_positions,
        'tournaments_last_year': total_tournaments, #recent_entries = all_entries[all_entries['year'] == last_year_cutoff]
        'cut_percentage_last_year': round(cut_percentage, 1) if not pd.isna(cut_percentage) else None,
        'last_3_avg_position': round(last_3_avg, 2) if not pd.isna(last_3_avg) else None,
        'last_5_avg_position': round(last_5_avg, 2) if not pd.isna(last_5_avg) else None,
        'last_10_avg_position': round(last_10_avg, 2) if not pd.isna(last_10_avg) else None,
        'top_5_count': round(top_5_count, 2),
        'top_10_count': round(top_10_count, 2),
        'top_20_count': round(top_20_count, 2),
        'top_5_percentage_last_year': round(top_5_percentage, 1),
        'top_10_percentage_last_year': round(top_10_percentage, 1),
        'top_20_percentage_last_year': round(top_20_percentage, 1)

    })


In [66]:
results

[{'player': 'A.J. Ewart',
  'total_tournaments': 5,
  'games_with_positions': 5,
  'tournaments_last_year': 1,
  'cut_percentage_last_year': 0.0,
  'last_3_avg_position': np.float64(40.33),
  'last_5_avg_position': np.float64(56.2),
  'last_10_avg_position': np.float64(56.2),
  'top_5_count': 0,
  'top_10_count': 0,
  'top_20_count': 0,
  'top_5_percentage_last_year': 0.0,
  'top_10_percentage_last_year': 0.0,
  'top_20_percentage_last_year': 0.0},
 {'player': 'A.J. Ewart (a)',
  'total_tournaments': 1,
  'games_with_positions': 1,
  'tournaments_last_year': 0,
  'cut_percentage_last_year': None,
  'last_3_avg_position': np.float64(80.0),
  'last_5_avg_position': np.float64(80.0),
  'last_10_avg_position': np.float64(80.0),
  'top_5_count': 0,
  'top_10_count': 0,
  'top_20_count': 0,
  'top_5_percentage_last_year': 0.0,
  'top_10_percentage_last_year': 0.0,
  'top_20_percentage_last_year': 0.0},
 {'player': 'Aaron Baddeley',
  'total_tournaments': 61,
  'games_with_positions': 60,
  '

In [67]:
# Convert to DataFrame
results_df = pd.DataFrame(results)

# Sort by best average position over last 5 games (lower is better)
results_df = results_df.sort_values('last_5_avg_position')

# Display results
top_n = 20
print(f"Top {top_n} Players by Average Finish Position")
print("=" * 90)
print(f"{'Player':<25} {'Tournaments':<12} {'Cut %':<8} {'Last 3 Avg':<12} {'Last 5 Avg':<12}")
print("-" * 90)

for idx, row in results_df.head(top_n).iterrows():
    #top_5 = f"{row['top_5_percentage']:.2f}" if row['top_5_percentage'] is not None else "N/A"
    last_3 = f"{row['last_3_avg_position']:.2f}" if row['last_3_avg_position'] is not None else "N/A"
    last_5 = f"{row['last_5_avg_position']:.2f}" if row['last_5_avg_position'] is not None else "N/A"
    cut_pct = f"{row['cut_percentage_last_year']:.1f}%" if row['cut_percentage_last_year'] is not None else "N/A"
    tournaments = f"{row['tournaments_last_year']}"
    
    print(f"{row['player']:<25} {tournaments:<12} {cut_pct:<8} {last_3:<12} {last_5:<12}")

print(f"\nTotal players analyzed: {len(results_df)}")
print(f"Cut percentage calculated over {current_year - last_year_cutoff + 1} year(s) of data")

# Save results to a new CSV
# results_df.to_csv('player_average_positions.csv', index=False)
print("Results saved to 'player_average_positions.csv'")


Top 20 Players by Average Finish Position
Player                    Tournaments  Cut %    Last 3 Avg   Last 5 Avg  
------------------------------------------------------------------------------------------
Scottie Scheffler         20           100.0%   2.67         2.60        
Jackson Koivun            1            100.0%   4.00         4.00        
Tommy Fleetwood           19           94.7%    3.00         5.60        
Hideki Matsuyama          23           87.0%    7.00         10.80       
Rory McIlroy              15           93.3%    16.33        11.60       
John Keefer               0            nan%     13.00        13.00       
Russell Henley            19           89.5%    15.33        13.40       
Si Woo Kim                29           75.9%    16.67        13.40       
Rickie Fowler             21           90.5%    18.33        13.60       
Pierceson Coody           13           69.2%    20.00        18.20       
Jake Knapp                22           72.7%    7.00 

In [68]:
results_df

,player,total_tournaments,games_with_positions,tournaments_last_year,cut_percentage_last_year,last_3_avg_position,last_5_avg_position,last_10_avg_position,top_5_count,top_10_count,top_20_count,top_5_percentage_last_year,top_10_percentage_last_year,top_20_percentage_last_year
1003,Scottie Scheffler,78,78,20,100.0,2.67,2.6,3.2,12,17,19,60.0,85.0,95.0
463,Jackson Koivun,1,1,1,100.0,4.00,4.0,4.0,1,1,1,100.0,100.0,100.0
1098,Tommy Fleetwood,67,67,19,94.7,3.00,5.6,16.4,7,8,13,36.8,42.1,68.4
436,Hideki Matsuyama,84,78,23,87.0,7.00,10.8,16.1,1,1,9,4.3,4.3,39.1
936,Rory McIlroy,61,61,15,93.3,16.33,11.6,21.7,5,8,12,33.3,53.3,80.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
585,Josh Goldenberg,1,1,1,0.0,80.00,80.0,80.0,0,0,0,0.0,0.0,0.0
586,Josh Radcliff,2,2,2,0.0,80.00,80.0,80.0,0,0,0,0.0,0.0,0.0
1140,Wells Williams (a),1,1,0,NaN,80.00,80.0,80.0,0,0,0,0.0,0.0,0.0
608,Justin Walters,2,1,1,0.0,80.00,80.0,80.0,0,0,0,0.0,0.0,0.0


In [69]:
#filtered_df = results_df[results_df['tournaments_last_year'] > 3]
filtered_df = results_df #[results_df['tournaments_last_year'] > 3]

In [70]:
filtered_df['last_3_avg_position_percentile'] = filtered_df['last_3_avg_position'].rank(pct=True,ascending=False)
filtered_df['last_5_avg_position_percentile'] = filtered_df['last_5_avg_position'].rank(pct=True,ascending=False)
filtered_df['last_10_avg_position_percentile'] = filtered_df['last_10_avg_position'].rank(pct=True,ascending=False)

filtered_df['cut_percentage_last_year_percentile'] = filtered_df['cut_percentage_last_year'].rank(pct=True,ascending=True)

filtered_df['top_5_percentage_last_year_percentile'] = filtered_df['top_5_percentage_last_year'].rank(pct=True,ascending=True)
filtered_df['top_10_percentage_last_year_percentile'] = filtered_df['top_10_percentage_last_year'].rank(pct=True,ascending=True)
filtered_df['top_20_percentage_last_year_percentile'] = filtered_df['top_20_percentage_last_year'].rank(pct=True,ascending=True)

In [71]:
# ok have 2 datasets; results and joined_df. 
# I need to join these 2 datasets together on player name 

In [72]:
filtered_df

,player,total_tournaments,games_with_positions,tournaments_last_year,cut_percentage_last_year,last_3_avg_position,last_5_avg_position,last_10_avg_position,top_5_count,top_10_count,...,top_5_percentage_last_year,top_10_percentage_last_year,top_20_percentage_last_year,last_3_avg_position_percentile,last_5_avg_position_percentile,last_10_avg_position_percentile,cut_percentage_last_year_percentile,top_5_percentage_last_year_percentile,top_10_percentage_last_year_percentile,top_20_percentage_last_year_percentile
1003,Scottie Scheffler,78,78,20,100.0,2.67,2.6,3.2,12,17,...,60.0,85.0,95.0,1.000000,1.000000,1.000000,0.943709,0.999157,0.999157,0.994941
463,Jackson Koivun,1,1,1,100.0,4.00,4.0,4.0,1,1,...,100.0,100.0,100.0,0.998314,0.999157,0.999157,0.943709,1.000000,1.000000,0.997892
1098,Tommy Fleetwood,67,67,19,94.7,3.00,5.6,16.4,7,8,...,36.8,42.1,68.4,0.999157,0.998314,0.994941,0.884106,0.996627,0.994098,0.990304
436,Hideki Matsuyama,84,78,23,87.0,7.00,10.8,16.1,1,1,...,4.3,4.3,39.1,0.996206,0.997470,0.995784,0.858444,0.908094,0.870152,0.968803
936,Rory McIlroy,61,61,15,93.3,16.33,11.6,21.7,5,8,...,33.3,53.3,80.0,0.989460,0.996627,0.989039,0.882450,0.994941,0.997470,0.994098
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
585,Josh Goldenberg,1,1,1,0.0,80.00,80.0,80.0,0,0,...,0.0,0.0,0.0,0.275717,0.252951,0.243676,0.227649,0.447302,0.431282,0.416526
586,Josh Radcliff,2,2,2,0.0,80.00,80.0,80.0,0,0,...,0.0,0.0,0.0,0.275717,0.252951,0.243676,0.227649,0.447302,0.431282,0.416526
1140,Wells Williams (a),1,1,0,NaN,80.00,80.0,80.0,0,0,...,0.0,0.0,0.0,0.275717,0.252951,0.243676,NaN,0.447302,0.431282,0.416526
608,Justin Walters,2,1,1,0.0,80.00,80.0,80.0,0,0,...,0.0,0.0,0.0,0.275717,0.252951,0.243676,0.227649,0.447302,0.431282,0.416526


In [73]:
joined_df 

,player,pos,to_par,year,week_of_season,tournament,course,avg,updated_avg,total_sg_t,...,SG_last_3_percentile,SG_last_5_percentile,SG_last_8_percentile,SG_last_13_percentile,SG_last_21_percentile,SG_last_34_percentile,SG_last_55_percentile,SG_last_89_percentile,dg_index_percentile,owgr_rank_percentile
0,A.J. Ewart,T28,-7,2026,6.0,WM Phoenix Open,TPC Scottsdale,0.924,0.924,3.694,...,0.9386,0.8729,0.8687,0.8683,0.8636,0.8611,0.8577,0.8569,0.3224,0.2494
1,A.J. Ewart (a),CUT,5,2022,23.0,RBC Canadian Open,TPC Toronto at Osprey Valley,None,-2,None,...,0.3590,0.3295,0.3018,0.2883,0.2862,0.2862,0.2858,0.2858,NaN,NaN
2,Aaron Baddeley,T72,2,2025,31.0,Wyndham Championship,Sedgefield Country Club,-1.204,-1.204,-4.816,...,0.7168,0.6524,0.5976,0.6549,0.6515,0.7391,0.7626,0.7433,NaN,NaN
3,Aaron Beverly,CUT,14,2022,7.0,The Genesis Invitational,The Riviera Country Club,None,-2,None,...,0.3590,0.3295,0.3018,0.2883,0.2862,0.2862,0.2858,0.2858,NaN,NaN
4,Aaron Cockerill,CUT,4,2025,28.0,Genesis Scottish Open,The Renaissance Club,None,-2,None,...,0.3590,0.7151,0.6978,0.6625,0.6397,0.6364,0.6309,0.6279,0.1385,0.1688
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1183,Zack Fischer,CUT,-1,2025,27.0,John Deere Classic,TPC Deere Run,None,-2,None,...,0.7744,0.7753,0.7559,0.7306,0.7117,0.7045,0.6978,0.6944,NaN,NaN
1184,Zander Lombard,CUT,2,2023,28.0,Genesis Scottish Open,The Renaissance Club,None,-2,None,...,0.3590,0.3295,0.3018,0.2883,0.2862,0.2862,0.2858,0.2858,0.1990,0.1511
1185,Zecheng Dou,T13,-11,2026,6.0,WM Phoenix Open,TPC Scottsdale,1.924,1.924,7.694,...,0.9604,0.9394,0.8443,0.8123,0.8590,0.7782,0.7715,0.7656,0.7103,0.5668
1186,Ángel Cabrera,CUT,11,2025,15.0,Masters Tournament,Augusta National Golf Club,None,-2,None,...,0.3590,0.3295,0.3018,0.2883,0.2862,0.2862,0.2858,0.2858,NaN,NaN


In [74]:
merged_dataset = pd.merge(joined_df, filtered_df, on='player', how='left')

In [75]:
merged_dataset = merged_dataset[merged_dataset['tournaments_last_year'] > 3]

In [76]:
merged_dataset

,player,pos,to_par,year,week_of_season,tournament,course,avg,updated_avg,total_sg_t,...,top_5_percentage_last_year,top_10_percentage_last_year,top_20_percentage_last_year,last_3_avg_position_percentile,last_5_avg_position_percentile,last_10_avg_position_percentile,cut_percentage_last_year_percentile,top_5_percentage_last_year_percentile,top_10_percentage_last_year_percentile,top_20_percentage_last_year_percentile
2,Aaron Baddeley,T72,2,2025,31.0,Wyndham Championship,Sedgefield Country Club,-1.204,-1.204,-4.816,...,0.0,0.0,0.0,0.562816,0.510118,0.624789,0.557947,0.447302,0.431282,0.416526
6,Aaron Rai,T73,1,2026,5.0,ATT Pebble Beach Pro-Am,Pebble Beach Golf Links,-2.613,-2.613,-10.451,...,10.0,10.0,35.0,0.876054,0.942664,0.913997,0.826987,0.962057,0.924536,0.959949
7,Aaron Wise,CUT,1,2026,4.0,Farmers Insurance Open,Torrey Pines Golf Course,None,-2,None,...,0.0,0.0,0.0,0.275717,0.569562,0.559022,0.547185,0.447302,0.431282,0.416526
10,Adam Hadwin,70,-7,2025,41.0,The RSM Classic,Sea Island Golf Club,None,-2,None,...,0.0,3.8,7.7,0.901349,0.778668,0.685497,0.692881,0.447302,0.864250,0.846965
12,Adam Schenk,CUT,7,2026,6.0,WM Phoenix Open,TPC Scottsdale,None,-2,None,...,8.7,13.0,13.0,0.817454,0.702361,0.822091,0.610099,0.947723,0.942664,0.871838
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1158,William Mouw,CUT,11,2026,6.0,WM Phoenix Open,TPC Scottsdale,None,-2,None,...,0.0,5.3,10.5,0.275717,0.685919,0.858347,0.585265,0.447302,0.883221,0.862985
1165,Wyndham Clark,T35,-6,2026,6.0,WM Phoenix Open,TPC Scottsdale,0.674,0.674,2.694,...,9.1,9.1,31.8,0.853710,0.869309,0.944351,0.839404,0.954469,0.913153,0.948145
1166,Xander Schauffele,T41,-5,2026,6.0,WM Phoenix Open,TPC Scottsdale,0.424,0.424,1.694,...,6.2,25.0,50.0,0.886594,0.952361,0.978921,0.943709,0.929595,0.978078,0.982293
1177,Zac Blair,T50,-4,2026,2.0,Sony Open in Hawaii,Waialae Country Club,0.218,0.218,0.871,...,0.0,13.3,20.0,0.980607,0.968803,0.921164,0.685430,0.447302,0.945194,0.901349


In [77]:
metric_col_1 = 'SG_last_1_percentile'
metric_col_2 = 'SG_last_2_percentile'
metric_col_3 = 'SG_last_3_percentile'
metric_col_4 = 'SG_last_5_percentile'
metric_col_5 = 'SG_last_8_percentile'
metric_col_6 = 'SG_last_13_percentile'
metric_col_7 = 'SG_last_21_percentile'
metric_col_8 = 'SG_last_34_percentile'
metric_col_9 = 'SG_last_55_percentile'
metric_col_10 = 'dg_index_percentile'
metric_col_11 = 'owgr_rank_percentile'
metric_col_12 = 'last_3_avg_position_percentile'
metric_col_13 = 'last_5_avg_position_percentile'
metric_col_14 = 'last_10_avg_position_percentile'
metric_col_15 = 'cut_percentage_last_year_percentile'
metric_col_16 = 'top_5_percentage_last_year_percentile'
metric_col_17 = 'top_10_percentage_last_year_percentile'
metric_col_18 = 'top_20_percentage_last_year_percentile'

In [78]:
# First Version, there was too much weight on most recent events. 
# This is a problem because some events have worse fields and this makes less competitive fields have an advantage. 

weight_1 = 0.882   # SG_last_1_percentile
weight_2 = 1.185   # SG_last_2_percentile
weight_3 = 1.512   # SG_last_3_percentile
weight_4 = 1.433   # SG_last_5_percentile
weight_5 = 0.911   # SG_last_8_percentile
weight_6 = 0.855   # SG_last_13_percentile
weight_7 = 0.702   # SG_last_21_percentile
weight_8 = 0.68   # SG_last_34_percentile
weight_9 = 0.8   # SG_last_55_percentile
weight_10 = 1.0  # dg_index_percentile
weight_11 = 0.5  # owgr_rank_percentile
weight_12 = 1.5  # last_3_avg_position_percentile
weight_13 = 1.1  # last_5_avg_position_percentile
weight_14 = 0.95  # last_10_avg_position_percentile
weight_15 = 0.56  # 'cut_percentage_last_year_percentile'
weight_16 = 1.3  # top_5_percentage_last_year_percentile
weight_17 = 0.98  # top_10_percentage_last_year_percentile
weight_18 = 0.923  # top_20_percentage_last_year_percentile



# Calculate total weight for normalization
total_weight = (weight_1 + weight_2 + weight_3 + weight_4 + weight_5 + 
               weight_6 + weight_7 + weight_8 + weight_9 + weight_10 + weight_10
               + weight_11 + weight_12 + weight_13 + weight_14 + weight_15 + weight_16 + weight_17 + weight_18 )


In [79]:
# Normalize weights so they sum to 1
norm_weight_1 = weight_1 / total_weight
norm_weight_2 = weight_2 / total_weight
norm_weight_3 = weight_3 / total_weight
norm_weight_4 = weight_4 / total_weight
norm_weight_5 = weight_5 / total_weight
norm_weight_6 = weight_6 / total_weight
norm_weight_7 = weight_7 / total_weight
norm_weight_8 = weight_8 / total_weight
norm_weight_9 = weight_9 / total_weight
norm_weight_10 = weight_10 / total_weight
norm_weight_11 = weight_11 / total_weight
norm_weight_12 = weight_12 / total_weight
norm_weight_13 = weight_13 / total_weight
norm_weight_14 = weight_14 / total_weight
norm_weight_15 = weight_15 / total_weight
norm_weight_16 = weight_16 / total_weight
norm_weight_17 = weight_17 / total_weight
norm_weight_18 = weight_18 / total_weight


print("Normalized weights (should sum to 1.0):")
print(f"{metric_col_1}: {norm_weight_1:.3f}")
print(f"{metric_col_2}: {norm_weight_2:.3f}")
print(f"{metric_col_3}: {norm_weight_3:.3f}")
print(f"{metric_col_4}: {norm_weight_4:.3f}")
print(f"{metric_col_5}: {norm_weight_5:.3f}")
print(f"{metric_col_6}: {norm_weight_6:.3f}")
print(f"{metric_col_7}: {norm_weight_7:.3f}")
print(f"{metric_col_8}: {norm_weight_8:.3f}")
print(f"{metric_col_9}: {norm_weight_9:.3f}")
print(f"{metric_col_10}: {norm_weight_10:.3f}")
print(f"{metric_col_11}: {norm_weight_11:.3f}")
print(f"{metric_col_12}: {norm_weight_12:.3f}")
print(f"{metric_col_13}: {norm_weight_13:.3f}")
print(f"{metric_col_14}: {norm_weight_14:.3f}")
print(f"{metric_col_15}: {norm_weight_15:.3f}")
print(f"{metric_col_16}: {norm_weight_16:.3f}")
print(f"{metric_col_17}: {norm_weight_17:.3f}")
print(f"{metric_col_18}: {norm_weight_18:.3f}")
print(f"Total: {sum([norm_weight_1, norm_weight_2, norm_weight_3, norm_weight_4, norm_weight_5, norm_weight_6, norm_weight_7, norm_weight_8, norm_weight_9, norm_weight_10, norm_weight_11, 
                     norm_weight_12, norm_weight_13, norm_weight_14, norm_weight_15, norm_weight_16, norm_weight_17, norm_weight_18]):.3f}")
print()


Normalized weights (should sum to 1.0):
SG_last_1_percentile: 0.047
SG_last_2_percentile: 0.063
SG_last_3_percentile: 0.081
SG_last_5_percentile: 0.076
SG_last_8_percentile: 0.049
SG_last_13_percentile: 0.046
SG_last_21_percentile: 0.037
SG_last_34_percentile: 0.036
SG_last_55_percentile: 0.043
dg_index_percentile: 0.053
owgr_rank_percentile: 0.027
last_3_avg_position_percentile: 0.080
last_5_avg_position_percentile: 0.059
last_10_avg_position_percentile: 0.051
cut_percentage_last_year_percentile: 0.030
top_5_percentage_last_year_percentile: 0.069
top_10_percentage_last_year_percentile: 0.052
top_20_percentage_last_year_percentile: 0.049
Total: 0.947



In [80]:
# Calculate weighted composite score for each player
weighted_scores = []

for index, row in merged_dataset.iterrows():
    # Get the values for each metric (handle missing values)
    val_1 = row[metric_col_1] if not pd.isna(row[metric_col_1]) else 0
    val_2 = row[metric_col_2] if not pd.isna(row[metric_col_2]) else 0
    val_3 = row[metric_col_3] if not pd.isna(row[metric_col_3]) else 0
    val_4 = row[metric_col_4] if not pd.isna(row[metric_col_4]) else 0
    val_5 = row[metric_col_5] if not pd.isna(row[metric_col_5]) else 0
    val_6 = row[metric_col_6] if not pd.isna(row[metric_col_6]) else 0
    val_7 = row[metric_col_7] if not pd.isna(row[metric_col_7]) else 0
    val_8 = row[metric_col_8] if not pd.isna(row[metric_col_8]) else 0
    val_9 = row[metric_col_9] if not pd.isna(row[metric_col_9]) else 0
    val_10 = row[metric_col_10] if not pd.isna(row[metric_col_10]) else 0
    val_11 = row[metric_col_11] if not pd.isna(row[metric_col_11]) else 0
    val_12 = row[metric_col_12] if not pd.isna(row[metric_col_12]) else 0
    val_13 = row[metric_col_13] if not pd.isna(row[metric_col_13]) else 0
    val_14 = row[metric_col_14] if not pd.isna(row[metric_col_14]) else 0
    val_15 = row[metric_col_15] if not pd.isna(row[metric_col_15]) else 0
    val_16 = row[metric_col_16] if not pd.isna(row[metric_col_16]) else 0
    val_17 = row[metric_col_17] if not pd.isna(row[metric_col_17]) else 0
    val_18 = row[metric_col_18] if not pd.isna(row[metric_col_18]) else 0
    
    # Calculate weighted score
    weighted_score = (val_1 * norm_weight_1 + 
                     val_2 * norm_weight_2 + 
                     val_3 * norm_weight_3 + 
                     val_4 * norm_weight_4 + 
                     val_5 * norm_weight_5 + 
                     val_6 * norm_weight_6 + 
                     val_7 * norm_weight_7 + 
                     val_8 * norm_weight_8 + 
                     val_9 * norm_weight_9 + 
                     val_10 * norm_weight_10 +
                     val_11 * norm_weight_11 +
                     val_12 * norm_weight_12 +
                     val_13 * norm_weight_13 +
                     val_14 * norm_weight_14 +
                     val_15 * norm_weight_15 + 
                     val_16 * norm_weight_16 +
                     val_17 * norm_weight_17 +
                     val_18 * norm_weight_18)
    
    weighted_scores.append(weighted_score)

In [81]:
# Add weighted scores to dataframe
merged_dataset['weighted_composite'] = weighted_scores

# Calculate final percentile rank manually
# Sort the weighted scores and assign ranks
df_sorted = merged_dataset.sort_values('weighted_composite', ascending=False)
df_sorted['rank'] = range(1, len(df_sorted) + 1)

df_sorted['composite_percentile_rank'] = df_sorted['weighted_composite'].rank(pct=True,ascending=True)


In [82]:
df_sorted = df_sorted.round(4)

In [83]:
composite_with_percentile = df_sorted[['player', 'weighted_composite', 'composite_percentile_rank']].copy()

In [84]:
composite_with_percentile.loc[composite_with_percentile['player'] == 'Ludvig Åberg', 'player'] = 'Ludvig Aberg'

# Enter updated player list here! 

In [85]:
# Link to Tier List for Golfers 
# tier_df = pd.read_csv('/Users/brandon/Desktop/Golf_Data/eastlake_list.csv') # Mac Link
tier_df = pd.read_csv(r"C:\Users\brand\OneDrive\Documents\pga_stats\upload_staging\player_list\genesis_list.csv")

In [86]:
composite_with_percentile

,player,weighted_composite,composite_percentile_rank
1004,Scottie Scheffler,0.9441,1.0000
1100,Tommy Fleetwood,0.9394,0.9958
943,Russell Henley,0.9338,0.9916
937,Rory McIlroy,0.9310,0.9874
731,Matt Fitzpatrick,0.9220,0.9833
...,...,...,...
910,Richard Hoey,0.3984,0.0209
272,Cristobal Del Solar,0.3877,0.0167
783,Michael La Sasso\n(a),0.3700,0.0126
152,Brendon Todd,0.3692,0.0084


In [87]:
df_sorted

,player,pos,to_par,year,week_of_season,tournament,course,avg,updated_avg,total_sg_t,...,last_3_avg_position_percentile,last_5_avg_position_percentile,last_10_avg_position_percentile,cut_percentage_last_year_percentile,top_5_percentage_last_year_percentile,top_10_percentage_last_year_percentile,top_20_percentage_last_year_percentile,weighted_composite,rank,composite_percentile_rank
1004,Scottie Scheffler,T3,-15,2026,6.0,WM Phoenix Open,TPC Scottsdale,2.924,2.924,11.694,...,1.0000,1.0000,1.0000,0.9437,0.9992,0.9992,0.9949,0.9441,1,1.0000
1100,Tommy Fleetwood,T4,-20,2026,5.0,ATT Pebble Beach Pro-Am,Pebble Beach Golf Links,2.637,2.637,10.549,...,0.9992,0.9983,0.9949,0.8841,0.9966,0.9941,0.9903,0.9394,2,0.9958
943,Russell Henley,T19,-15,2026,5.0,ATT Pebble Beach Pro-Am,Pebble Beach Golf Links,1.125,1.125,4.499,...,0.9916,0.9945,0.9983,0.8684,0.9924,0.9966,0.9903,0.9338,3,0.9916
937,Rory McIlroy,T14,-17,2026,5.0,ATT Pebble Beach Pro-Am,Pebble Beach Golf Links,1.888,1.888,7.550,...,0.9895,0.9966,0.9890,0.8825,0.9949,0.9975,0.9941,0.9310,4,0.9874
731,Matt Fitzpatrick,9,-13,2026,6.0,WM Phoenix Open,TPC Scottsdale,2.424,2.424,9.694,...,0.9663,0.9768,0.9941,0.8551,0.9621,0.9781,0.9599,0.9220,5,0.9833
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
910,Richard Hoey,T25,-11,2025,9.0,Cognizant Classic in The Palm Beaches,PGA National Resort,None,-2,None,...,0.7546,0.7167,0.8769,0.6101,0.4473,0.4313,0.4165,0.3984,235,0.0209
272,Cristobal Del Solar,CUT,-6,2025,41.0,The RSM Classic,Sea Island Golf Club,None,-2,None,...,0.2757,0.2530,0.5253,0.5000,0.4473,0.4313,0.4165,0.3877,236,0.0167
783,Michael La Sasso\n(a),T44,-11,2025,30.0,3M Open,TPC Twin Cities,None,-2,None,...,0.6931,0.6180,0.6029,0.4818,0.4473,0.4313,0.4165,0.3700,237,0.0126
152,Brendon Todd,CUT,3,2025,41.0,The RSM Classic,Sea Island Golf Club,None,-2,None,...,0.2757,0.5051,0.6771,0.5190,0.4473,0.4313,0.4165,0.3692,238,0.0084


In [88]:
playerlist_metrics = pd.merge(
    tier_df, 
    composite_with_percentile, 
    on='player', 
    how='left'  # Keep all tier players, even if no composite score
)

In [89]:
# STEP 6: Analyze by tier
print("\n=== TOURNAMENT TIER ANALYSIS ===")
print("Best players in each Tier (by composite score):")
print("=" * 70)

for tier_num in sorted(playerlist_metrics['Tier'].unique()):
    tier_players = playerlist_metrics[playerlist_metrics['Tier'] == tier_num].sort_values('weighted_composite', ascending=False)
    
    print(f"\nTIER {tier_num} (Top 5):")
    print(f"{'Rank':<4} {'Player':<25} {'Composite':<12} {'Percentile':<12}")
    print("-" * 55)
    
    for i, (_, player) in enumerate(tier_players.iterrows(), 1):
        comp_score = f"{player['weighted_composite']:.4f}" if not pd.isna(player['weighted_composite']) else "N/A"
        pct_rank = f"{player['composite_percentile_rank']:.4f}%" if not pd.isna(player['composite_percentile_rank']) else "N/A"
        print(f"{i:<4} {player['player']:<25} {comp_score:<12} {pct_rank:<12}")

# STEP 7: Optimal picks for your competition
print("\n=== OPTIMAL PICKS FOR COMPETITION ===")
print("Best player from each tier:")
print("=" * 60)

optimal_picks = []
total_composite = 0

for tier_num in sorted(playerlist_metrics['Tier'].unique()):
    tier_players = playerlist_metrics[playerlist_metrics['Tier'] == tier_num].sort_values('weighted_composite', ascending=False)
    best_player = tier_players.iloc[0]
    
    optimal_picks.append({
        'Tier': tier_num,
        'player': best_player['player'],
        'composite_score': best_player['weighted_composite'],
        'percentile_rank': best_player['composite_percentile_rank']
    })
    
    total_composite += best_player['weighted_composite']
    
    comp_score = f"{best_player['weighted_composite']:.4f}" if not pd.isna(best_player['weighted_composite']) else "N/A"
    pct_rank = f"{best_player['composite_percentile_rank']:.4f}%" if not pd.isna(best_player['composite_percentile_rank']) else "N/A"
    
    print(f"Tier {tier_num}: {best_player['player']:<25} (Score: {comp_score}, Rank: {pct_rank})")

print(f"\nTotal Team Composite Score: {total_composite:.4f}")
print(f"Average Team Composite Score: {total_composite/6:.4f}")



=== TOURNAMENT TIER ANALYSIS ===
Best players in each Tier (by composite score):

TIER 1 (Top 5):
Rank Player                    Composite    Percentile  
-------------------------------------------------------
1    Tommy Fleetwood           0.9394       0.9958%     
2    Rory McIlroy              0.9310       0.9874%     
3    Xander Schauffele         0.8959       0.9038%     

TIER 2 (Top 5):
Rank Player                    Composite    Percentile  
-------------------------------------------------------
1    Russell Henley            0.9338       0.9916%     
2    Ben Griffin               0.9190       0.9749%     
3    Hideki Matsuyama          0.9180       0.9707%     
4    Si Woo Kim                0.9156       0.9582%     
5    Viktor Hovland            0.9144       0.9498%     
6    Chris Gotterup            0.9108       0.9414%     
7    Patrick Cantlay           0.9106       0.9372%     
8    Cameron Young             0.9088       0.9289%     
9    Collin Morikawa           

In [90]:
df_sorted

,player,pos,to_par,year,week_of_season,tournament,course,avg,updated_avg,total_sg_t,...,last_3_avg_position_percentile,last_5_avg_position_percentile,last_10_avg_position_percentile,cut_percentage_last_year_percentile,top_5_percentage_last_year_percentile,top_10_percentage_last_year_percentile,top_20_percentage_last_year_percentile,weighted_composite,rank,composite_percentile_rank
1004,Scottie Scheffler,T3,-15,2026,6.0,WM Phoenix Open,TPC Scottsdale,2.924,2.924,11.694,...,1.0000,1.0000,1.0000,0.9437,0.9992,0.9992,0.9949,0.9441,1,1.0000
1100,Tommy Fleetwood,T4,-20,2026,5.0,ATT Pebble Beach Pro-Am,Pebble Beach Golf Links,2.637,2.637,10.549,...,0.9992,0.9983,0.9949,0.8841,0.9966,0.9941,0.9903,0.9394,2,0.9958
943,Russell Henley,T19,-15,2026,5.0,ATT Pebble Beach Pro-Am,Pebble Beach Golf Links,1.125,1.125,4.499,...,0.9916,0.9945,0.9983,0.8684,0.9924,0.9966,0.9903,0.9338,3,0.9916
937,Rory McIlroy,T14,-17,2026,5.0,ATT Pebble Beach Pro-Am,Pebble Beach Golf Links,1.888,1.888,7.550,...,0.9895,0.9966,0.9890,0.8825,0.9949,0.9975,0.9941,0.9310,4,0.9874
731,Matt Fitzpatrick,9,-13,2026,6.0,WM Phoenix Open,TPC Scottsdale,2.424,2.424,9.694,...,0.9663,0.9768,0.9941,0.8551,0.9621,0.9781,0.9599,0.9220,5,0.9833
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
910,Richard Hoey,T25,-11,2025,9.0,Cognizant Classic in The Palm Beaches,PGA National Resort,None,-2,None,...,0.7546,0.7167,0.8769,0.6101,0.4473,0.4313,0.4165,0.3984,235,0.0209
272,Cristobal Del Solar,CUT,-6,2025,41.0,The RSM Classic,Sea Island Golf Club,None,-2,None,...,0.2757,0.2530,0.5253,0.5000,0.4473,0.4313,0.4165,0.3877,236,0.0167
783,Michael La Sasso\n(a),T44,-11,2025,30.0,3M Open,TPC Twin Cities,None,-2,None,...,0.6931,0.6180,0.6029,0.4818,0.4473,0.4313,0.4165,0.3700,237,0.0126
152,Brendon Todd,CUT,3,2025,41.0,The RSM Classic,Sea Island Golf Club,None,-2,None,...,0.2757,0.5051,0.6771,0.5190,0.4473,0.4313,0.4165,0.3692,238,0.0084


In [1]:
player_record = df_sorted.loc[df_sorted['player'] == 'Jake Knapp']
print(player_record)

NameError: name 'df_sorted' is not defined

In [92]:
# Returns a DataFrame containing only the row for that player
player_record = df.loc[df['Player'] == 'LeBron James']


KeyError: 'Player'

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

# Jupyter-optimized plotting settings
%matplotlib inline
plt.rcParams['figure.figsize'] = (15, 10)
plt.rcParams['font.size'] = 10
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
sns.set_style("whitegrid")
sns.set_palette("husl")

def clean_position(pos):
    """Convert position to numeric, handling ties like 'T5' -> 5"""
    if pd.isna(pos):
        return np.nan
    
    pos_str = str(pos).upper().strip()
    
    # Handle cuts - exclude from analysis
    if pos_str in ['CUT', 'MC', 'WD', 'DQ']:
        return np.nan
    
    # Handle tied positions - remove 'T' and convert to int
    if pos_str.startswith('T'):
        try:
            return int(pos_str[1:])
        except ValueError:
            return np.nan
    
    # Handle regular positions
    try:
        return int(pos_str)
    except ValueError:
        return np.nan

def display_summary_table(df, title, max_rows=10):
    """Display a nicely formatted table in Jupyter"""
    # Check which columns exist for formatting
    format_dict = {}
    if 'avg_position' in df.columns:
        format_dict['avg_position'] = '{:.1f}'
    if 'avg_score' in df.columns:
        format_dict['avg_score'] = '{:.1f}'
    if 'recent_avg_pos' in df.columns:
        format_dict['recent_avg_pos'] = '{:.1f}'
    if 'recent_avg_score' in df.columns:
        format_dict['recent_avg_score'] = '{:.1f}'
    
    styled_df = df.head(max_rows).style.set_caption(title)\
        .format(format_dict)\
        .set_table_styles([
            {'selector': 'caption', 'props': [('font-size', '16px'), ('font-weight', 'bold')]},
            {'selector': 'th', 'props': [('background-color', '#f0f0f0'), ('font-weight', 'bold')]},
            {'selector': 'td', 'props': [('padding', '8px'), ('text-align', 'center')]}
        ])
    
    # Add background gradient if top_10_count column exists
    if 'top_10_count' in df.columns:
        styled_df = styled_df.background_gradient(subset=['top_10_count'], cmap='Greens')
    elif 'Top 10s' in df.columns:
        styled_df = styled_df.background_gradient(subset=['Top 10s'], cmap='Greens')
    
    display(styled_df)

def analyze_tournament(df, tournament_name, show_plots=True, show_tables=True):
    """
    Jupyter-optimized tournament analysis with interactive displays
    """
    
    # Filter data for the specific tournament
    tournament_data = df[df['tournament'].str.contains(tournament_name, case=False, na=False)].copy()
    
    if tournament_data.empty:
        print(f"❌ No data found for tournament containing '{tournament_name}'")
        print("\n📋 Available tournaments:")
        available_tournaments = pd.DataFrame({'Available Tournaments': sorted(df['tournament'].unique())})
        display(available_tournaments.head(20))
        return None
    
    # Clean position data to handle ties
    tournament_data['position_numeric'] = tournament_data['pos'].apply(clean_position)
    
    # Remove rows with invalid positions (cuts, etc.)
    valid_data = tournament_data.dropna(subset=['position_numeric', 'to_par']).copy()
    
    # Display tournament info
    display(HTML(f"""
    <div style='background-color: #e8f5e8; padding: 15px; border-radius: 10px; margin: 10px 0;'>
        <h2 style='color: #2e7d32; margin: 0;'>🏆 {tournament_name.upper()} ANALYSIS</h2>
        <p style='margin: 5px 0; font-size: 14px;'>
            📅 <strong>Data Span:</strong> {valid_data['year'].min()} - {valid_data['year'].max()}<br>
            📊 <strong>Total Valid Records:</strong> {len(valid_data):,}<br>
            🚫 <strong>Records Excluded (Cuts/WD):</strong> {len(tournament_data) - len(valid_data):,}
        </p>
    </div>
    """))
    
    if show_plots:
        # Create optimized plots for Jupyter
        fig = plt.figure(figsize=(16, 12))
        
        # 1. WINNING SCORES BY YEAR
        ax1 = plt.subplot(2, 2, 1)
        yearly_winners = valid_data[valid_data['position_numeric'] == 1].groupby('year').agg({
            'to_par': 'mean',
            'player': 'first'
        }).reset_index()
        
        if not yearly_winners.empty:
            ax1.plot(yearly_winners['year'], yearly_winners['to_par'], 
                    marker='o', linewidth=3, markersize=8, color='#2e7d32', alpha=0.8)
            
            # Add trend line
            if len(yearly_winners) > 1:
                z = np.polyfit(yearly_winners['year'], yearly_winners['to_par'], 1)
                p = np.poly1d(z)
                ax1.plot(yearly_winners['year'], p(yearly_winners['year']), 
                        "--", alpha=0.6, color='#ff6b35', linewidth=2, label='Trend')
                ax1.legend()
            
            ax1.set_title('🏆 Winning Scores by Year', fontweight='bold', fontsize=12)
            ax1.set_xlabel('Year')
            ax1.set_ylabel('Winning Score (to par)')
            ax1.axhline(y=0, color='red', linestyle='--', alpha=0.5, label='Even Par')
            
            # Annotate recent winner
            if len(yearly_winners) > 0:
                recent = yearly_winners.iloc[-1]
                ax1.annotate(f"{recent['player']}\n{recent['to_par']:+.0f}", 
                            xy=(recent['year'], recent['to_par']), 
                            xytext=(10, 10), textcoords='offset points',
                            bbox=dict(boxstyle='round,pad=0.3', facecolor='#ffeb3b', alpha=0.8),
                            fontsize=9, fontweight='bold')
        
        # 2. TOP 10 DISTRIBUTION BY YEAR
        ax2 = plt.subplot(2, 2, 2)
        top_10_by_year = valid_data[valid_data['position_numeric'] <= 10].groupby('year').size()
        
        if not top_10_by_year.empty:
            bars = ax2.bar(top_10_by_year.index, top_10_by_year.values, 
                          alpha=0.7, color='#1976d2', edgecolor='white', linewidth=1)
            ax2.set_title('📈 Top 10 Finishes by Year', fontweight='bold', fontsize=12)
            ax2.set_xlabel('Year')
            ax2.set_ylabel('Count of Top 10 Finishes')
            
            # Add value labels on bars
            for bar in bars:
                height = bar.get_height()
                ax2.text(bar.get_x() + bar.get_width()/2., height + 0.1,
                        f'{int(height)}', ha='center', va='bottom', fontweight='bold')
        
        # 3. CONSISTENT PERFORMERS
        ax3 = plt.subplot(2, 2, 3)
        top_10_data = valid_data[valid_data['position_numeric'] <= 10]
        player_performance = top_10_data.groupby('player').agg({
            'position_numeric': ['count', 'mean'],
            'to_par': 'mean',
            'year': ['min', 'max']
        }).round(2)
        
        player_performance.columns = ['top_10_count', 'avg_position', 'avg_score', 'first_year', 'last_year']
        player_performance = player_performance.reset_index()
        consistent_performers = player_performance[player_performance['top_10_count'] >= 2]\
            .sort_values('top_10_count', ascending=True).tail(12)
        
        if not consistent_performers.empty:
            colors = plt.cm.viridis(np.linspace(0, 1, len(consistent_performers)))
            bars = ax3.barh(range(len(consistent_performers)), consistent_performers['top_10_count'], 
                           color=colors, alpha=0.8, edgecolor='white', linewidth=1)
            ax3.set_yticks(range(len(consistent_performers)))
            ax3.set_yticklabels([name[:20] + '...' if len(name) > 20 else name 
                                for name in consistent_performers['player']], fontsize=9)
            ax3.set_title('🎯 Most Consistent Performers\n(Multiple Top 10s)', fontweight='bold', fontsize=12)
            ax3.set_xlabel('Number of Top 10 Finishes')
            
            # Add value labels
            for i, bar in enumerate(bars):
                width = bar.get_width()
                ax3.text(width + 0.1, bar.get_y() + bar.get_height()/2, 
                        f'{int(width)}', ha='left', va='center', fontweight='bold')
        
        # 4. SCORING BY POSITION
        ax4 = plt.subplot(2, 2, 4)
        position_scores = valid_data[valid_data['position_numeric'] <= 20]\
            .groupby('position_numeric')['to_par'].mean()
        
        if not position_scores.empty:
            bars = ax4.bar(position_scores.index, position_scores.values, 
                          alpha=0.7, color='#e91e63', edgecolor='white', linewidth=1)
            ax4.set_title('📊 Average Score by Position\n(Top 20)', fontweight='bold', fontsize=12)
            ax4.set_xlabel('Finishing Position')
            ax4.set_ylabel('Average Score (to par)')
            ax4.axhline(y=0, color='red', linestyle='--', alpha=0.5, label='Even Par')
            ax4.legend()
            
            # Highlight winner's position
            if len(position_scores) > 0:
                bars[0].set_color('#ffc107')
                bars[0].set_alpha(1.0)
        
        plt.tight_layout()
        plt.show()
    
    # DETAILED STATISTICS TABLES
    if show_tables:
        # Winning scores summary
        if not yearly_winners.empty:
            winning_stats = pd.DataFrame({
                'Metric': ['Average Winning Score', 'Best Winning Score (Most Under Par)', 
                          'Worst Winning Score (Least Under Par)', 'Most Recent Winner'],
                'Value': [f"{yearly_winners['to_par'].mean():.2f}",
                         f"{yearly_winners['to_par'].min():.2f}",
                         f"{yearly_winners['to_par'].max():.2f}",
                         f"{yearly_winners.iloc[-1]['player']} ({yearly_winners.iloc[-1]['to_par']:+.0f})"]
            })
            
            display(HTML("<h3 style='color: #2e7d32;'>🏆 Winning Score Statistics</h3>"))
            styled_winning_stats = winning_stats.style.set_table_styles([
                {'selector': 'th', 'props': [('background-color', '#e8f5e8'), ('font-weight', 'bold')]},
                {'selector': 'td', 'props': [('padding', '8px')]}
            ])
            display(styled_winning_stats)
        
        # Consistent performers table
        if not consistent_performers.empty:
            display(HTML("<h3 style='color: #1976d2;'>🎯 Most Consistent Performers (2+ Top 10s)</h3>"))
            display_table = consistent_performers[['player', 'top_10_count', 'avg_position', 'avg_score', 'first_year', 'last_year']].copy()
            display_table['years_active'] = display_table['first_year'].astype(int).astype(str) + '-' + display_table['last_year'].astype(int).astype(str)
            display_table = display_table[['player', 'top_10_count', 'avg_position', 'avg_score', 'years_active']].sort_values('top_10_count', ascending=False)
            display_table.columns = ['Player', 'Top 10s', 'Avg Position', 'Avg Score', 'Years Active']
            
            display_summary_table(display_table, "Most Consistent Performers")
        
        # Recent form table
        recent_years = valid_data['year'].max() - 2
        recent_top_10 = valid_data[
            (valid_data['year'] >= recent_years) & 
            (valid_data['position_numeric'] <= 10)
        ].groupby('player').agg({
            'position_numeric': ['count', 'mean'],
            'to_par': 'mean'
        }).round(2)
        
        recent_top_10.columns = ['recent_top_10s', 'recent_avg_pos', 'recent_avg_score']
        recent_top_10 = recent_top_10.reset_index()
        recent_top_10 = recent_top_10[recent_top_10['recent_top_10s'] >= 1]\
            .sort_values('recent_top_10s', ascending=False)
        
        if not recent_top_10.empty:
            display(HTML("<h3 style='color: #e91e63;'>🔥 Recent Form (Last 3 Years)</h3>"))
            recent_display = recent_top_10[['player', 'recent_top_10s', 'recent_avg_pos', 'recent_avg_score']].copy()
            recent_display.columns = ['Player', 'Top 10s', 'Avg Position', 'Avg Score']
            display_summary_table(recent_display, "Recent Top 10 Performers")
    
    return {
        'tournament_data': valid_data,
        'yearly_winners': yearly_winners,
        'consistent_performers': consistent_performers if 'consistent_performers' in locals() else pd.DataFrame(),
        'recent_performers': recent_top_10 if 'recent_top_10' in locals() else pd.DataFrame()
    }

def quick_tournament_overview(df, tournaments_list):
    """Quick overview of multiple tournaments"""
    overview_data = []
    
    for tournament in tournaments_list:
        tournament_data = df[df['tournament'].str.contains(tournament, case=False, na=False)].copy()
        
        if tournament_data.empty:
            continue
            
        tournament_data['position_numeric'] = tournament_data['pos'].apply(clean_position)
        valid_data = tournament_data.dropna(subset=['position_numeric', 'to_par'])
        
        if valid_data.empty:
            continue
        
        # Calculate key stats
        winners = valid_data[valid_data['position_numeric'] == 1]
        avg_winning_score = winners['to_par'].mean() if not winners.empty else np.nan
        total_years = valid_data['year'].nunique()
        total_players = valid_data['player'].nunique()
        
        overview_data.append({
            'Tournament': tournament,
            'Years of Data': total_years,
            'Unique Players': total_players,
            'Avg Winning Score': f"{avg_winning_score:.1f}" if not pd.isna(avg_winning_score) else "N/A",
            'Total Records': len(valid_data)
        })
    
    if overview_data:
        overview_df = pd.DataFrame(overview_data)
        display(HTML("<h2 style='color: #2e7d32;'>🏌️ Tournament Overview</h2>"))
        styled_overview = overview_df.style.set_table_styles([
            {'selector': 'th', 'props': [('background-color', '#e8f5e8'), ('font-weight', 'bold')]},
            {'selector': 'td', 'props': [('padding', '8px'), ('text-align', 'center')]}
        ])
        display(styled_overview)

# JUPYTER NOTEBOOK HELPER FUNCTIONS
def list_available_tournaments(df, search_term=""):
    """List all available tournaments with search functionality"""
    tournaments = df['tournament'].unique()
    
    if search_term:
        tournaments = [t for t in tournaments if search_term.lower() in t.lower()]
    
    tournament_df = pd.DataFrame({
        'Available Tournaments': sorted(tournaments),
        'Record Count': [len(df[df['tournament'] == t]) for t in sorted(tournaments)]
    })
    
    display(HTML(f"<h3>🏌️ Available Tournaments {f'(containing \"{search_term}\")' if search_term else ''}</h3>"))
    styled_tournaments = tournament_df.style.set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#f0f0f0'), ('font-weight', 'bold')]},
        {'selector': 'td', 'props': [('padding', '8px')]}
    ])
    display(styled_tournaments)
    
    return tournament_df

# USAGE EXAMPLES FOR JUPYTER
print("""
🏌️ GOLF TOURNAMENT ANALYSIS - JUPYTER OPTIMIZED

Quick Start Commands:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# 1. List available tournaments
list_available_tournaments(df)

# 2. Search for specific tournaments
list_available_tournaments(df, "Masters")

# 3. Analyze a specific tournament
results = analyze_tournament(df, "Masters")

# 4. Quick overview of multiple tournaments
quick_tournament_overview(df, ["Masters", "PGA", "U.S. Open"])

# 5. Analyze without plots (tables only)
results = analyze_tournament(df, "Masters", show_plots=False)

# 6. Show plots only (no tables)
results = analyze_tournament(df, "Masters", show_tables=False)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

In [ ]:
# Need to create a program that prints out the predicted score for the tie breaker for each round 

In [ ]:
results = analyze_tournament(df, "The Genesis Invitational")

In [ ]:
'''
Model improvement ideas:
- Pull in historical finishing position for the current event; add it as a metric 
- are there other metrics that could be pulled in; driving accurary; breaking out SGs 
- Fix Ludgid Aberg's name 

'''

In [ ]:
'''
Next Steps: DONE 8-7-25 
- create a new column that combines all of the percentile ranks into a single score 
- ask AI what makes the most sense 
- figure out how to use golfer tiers to find the best value; what will the form look like? 

''' 

In [ ]:
'''
# Ensure year and week_of_season are integers (no decimals)
df['year'] = df['year'].astype(int)
df['week_of_season'] = df['week_of_season'].astype(int)

# Create synthetic event date
df['event_date'] = pd.to_datetime(df['year'].astype(str) + df['week_of_season'].astype(str) + '1', format='%G%V%u')

# Sort by golfer and event order
df = df.sort_values(['player', 'event_date'])

# Rolling averages
df['SG_last_3'] = df.groupby('player')['updated_avg'].transform(lambda x: x.rolling(3, min_periods=1).mean())
df['SG_last_6'] = df.groupby('player')['updated_avg'].transform(lambda x: x.rolling(6, min_periods=1).mean())
df['SG_last_10'] = df.groupby('player')['updated_avg'].transform(lambda x: x.rolling(10, min_periods=1).mean())

# Get latest event per golfer
latest = df.groupby('player').tail(1)

# Select desired output
output_df = latest[['player', 'SG_last_3', 'SG_last_6', 'SG_last_10']]
'''

In [ ]:
'''
# Sort descending (best performers at top)
output_df = output_df.sort_values(by='SG_last_3', ascending=False)
print(output_df)
'''